# LingoPrism – Huấn luyện YOLOv10 (78 lớp)

**Cách chạy**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**. Khi được hỏi quyền Google Drive thì bấm cho phép.
3. Giữ tab mở (Colab miễn phí ngắt khi bỏ tab quá lâu). Tổng thời gian khoảng **2–3 giờ**.
4. Ô cuối tự tải về `lingoprism-yolov10n.zip` → gửi file đó lại để đăng ký model vào server.

**Nếu bị ngắt giữa chừng:** chỉ cần *Run all* lại – checkpoint nằm trên Drive (`MyDrive/lingoprism_runs`), notebook sẽ train tiếp chứ không train lại từ đầu.


In [ ]:
# 1. Kiểm tra GPU, lấy mã nguồn, cài thư viện
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
if not os.path.exists('/content/LingoPrism'):
    !git clone -q https://github.com/GPTii/LingoPrism.git /content/LingoPrism
else:
    !git -C /content/LingoPrism pull -q
%cd /content/LingoPrism
!pip install -q ultralytics fiftyone pyyaml onnx onnxslim onnxruntime
import torch
assert torch.cuda.is_available(), 'Chưa bật GPU: Runtime → Change runtime type → T4 GPU'


In [ ]:
# 2. Cấu hình – có thể chỉnh
PER_CLASS = 150          # ảnh tối đa mỗi lớp mỗi nguồn (COCO / Open Images)
EPOCHS    = 50
MODEL     = 'yolov10n.pt'  # bản nhẹ, chạy được cả trên điện thoại
NAME      = 'lingoprism-yolov10n'

from google.colab import drive
drive.mount('/content/drive')
DS   = '/content/lingoprism_ds'                    # dataset ở ổ Colab (nhanh, không tốn Drive)
RUNS = '/content/drive/MyDrive/lingoprism_runs'    # checkpoint + kết quả ở Drive (giữ khi bị ngắt)
os.makedirs(RUNS, exist_ok=True)


In [ ]:
# 3. Dựng dataset từ COCO-2017 + Open Images V7 (bỏ qua nếu đã có)
if not os.path.exists(f'{DS}/data.yaml'):
    !python AI/scripts/build_dataset.py --per-class {PER_CLASS} --out {DS}
else:
    print('Dataset đã có:', DS)


In [ ]:
# 4. Train (tự train tiếp nếu có checkpoint) + đánh giá trên tập test + export ONNX & TFLite
!python AI/scripts/train_eval_export.py --data {DS}/data.yaml --model {MODEL} --epochs {EPOCHS} --project {RUNS} --name {NAME}


In [ ]:
# 5. Kết quả
import json
from IPython.display import Image, display
run = f'{RUNS}/{NAME}'
m = json.load(open(f'{run}/report/metrics.json'))
print(f"Precision {m['precision']:.3f} | Recall {m['recall']:.3f} | mAP50 {m['map50']:.3f} | mAP50-95 {m['map50_95']:.3f}")
display(Image(f'{run}/results.png', width=900))
display(Image(f'{run}/test/confusion_matrix_normalized.png', width=900))


In [ ]:
# 6. Nén & tải về (gửi file zip này lại cho server)
import glob, shutil, zipfile
from google.colab import files
out = f'/content/{NAME}.zip'
keep = ['model_card.json', 'results.csv', 'results.png', 'args.yaml']
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in keep:
        if os.path.exists(f'{run}/{f}'):
            z.write(f'{run}/{f}', f)
    for f in glob.glob(f'{run}/weights/best*') + glob.glob(f'{run}/weights/best_saved_model/*float16.tflite'):
        if os.path.isfile(f):
            z.write(f, 'weights/' + os.path.basename(f))
    for folder in ('report', 'test'):
        for f in glob.glob(f'{run}/{folder}/*'):
            if os.path.isfile(f):
                z.write(f, f'{folder}/' + os.path.basename(f))
print(out, round(os.path.getsize(out) / 1e6, 1), 'MB')
files.download(out)
